# Synthesis and Capture

We'll now examine how to generate and capture signals with the RF hardware integrated into the RFSoC. 

In [ ]:
from acadia.system import Acadia
from acadia.channel import Channel

In [ ]:
acadia = Acadia()
acadia.attach()

In [ ]:
acadia.configure_clocks(reference="external")
Channel.reset_clk_distribution()

In [ ]:
acadia.get_clock_status()

In [ ]:
Channel.RFDC_status()

In [ ]:
acadia = Acadia()

capture_time = 5000e-9
pulse_time = 1000e-9
pulse2_time = 1000e-9

pulse_channel = acadia.DAC(1)
capture_channel = acadia.ADC(1)

pulse_length = pulse_channel.seconds_to_samples(pulse_time)
pulse2_length = pulse_channel.seconds_to_samples(pulse2_time)
capture_length = capture_channel.seconds_to_samples(capture_time)

pulse_memory = acadia.DACArray[pulse_channel.num](size=pulse_channel.seconds_to_bytes(pulse_time))
pulse2_memory = acadia.DACArray[pulse_channel.num](size=pulse_channel.seconds_to_bytes(pulse2_time))
capture_memory = acadia.PLDDR0Array(size=capture_channel.seconds_to_bytes(capture_time))

# Create a sequence for the sequencer
@acadia.sequence
def sequence(a):
    with a.synchronizer():
        a.generate(pulse_channel, pulse_memory)
        a.generate(pulse_channel, pulse2_memory)
        a.capture(capture_channel, capture_memory)
    a.active_sequencer.nop()
    a.active_sequencer.halt()

# Instruct the PS to create a 100ns cosine pulse and load it into memory
def program():   
    import numpy as np
    import time
    
    # Load the pulses into DAC memory
    pulse = np.ones(pulse_length, dtype=np.complex64)
    pulse_samples = pulse_channel.to_samples(pulse)
    acadia.memcpy(pulse_samples, pulse_memory)
    
    pulse2 = (1/2) - (1/2)*np.cos(2*np.pi*np.arange(pulse2_length)/pulse2_length, dtype=np.complex64)
    pulse2_samples = pulse_channel.to_samples(pulse2)
    acadia.memcpy(pulse2_samples, pulse2_memory)
    
    # Set up the channel properties
    pulse_channel.set_nyquist_zone(2)
    pulse_channel.configure_nco(frequency=1205e6)
    pulse_channel.set_vop(40000)
    capture_channel.set_nyquist_zone(2)
    capture_channel.set_dsa(0)
    
    # Clear the DDR array
    zeros = np.zeros(capture_length, dtype=np.complex64)
    zero_samples = capture_channel.to_samples(zeros)
    acadia.memcpy(zero_samples, capture_memory) 
    time.sleep(0.1) # Give the memory a moment to load
    
    # Configure the ADC switch
    acadia.configure_adc_switch()

    # Reset and run the sequencer
    acadia.sequencer_reset()
    acadia.sequencer_run(sequence)
    time.sleep(0.1)
    acadia.sequencer_halt()
    

In [ ]:
acadia.compile_all()

In [ ]:
acadia.assemble(load=True)

In [ ]:
program()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

time_per_sample = capture_time / capture_channel.seconds_to_samples(capture_time)
trace = np.frombuffer(capture_memory.memory, dtype=np.int16)
plt.plot(np.arange(0, capture_time, time_per_sample), trace[::2])